In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@databricksuk2025.dfs.core.windows.net/products")

In [0]:
df = df.drop("_rescued_data")

In [0]:
df.display()

### Functions

In [0]:
# View creation

df.createOrReplaceTempView("products")

In [0]:
%sql

-- Function creation

Create or Replace function databricks_cata.bronze.discount_func(p_price double)
  RETURNS DOUBLE
  language SQL
  Return p_price * 0.90

In [0]:
%sql
Select product_ID, price, databricks_cata.bronze.discount_func(price) as discounted_price from products

In [0]:
# Adding a column to the DF using the function, it requires to use the "Expression"

df = df.withColumn("Discounted_price",expr("databricks_cata.bronze.discount_func(price)"))
df.display()

In [0]:
# Function in SQL using Python in it

%sql
Create or Replace function databricks_cata.bronze.upper_func(p_brand string)
  Returns string
  Language Python
  As
  $$
    Return p_brand.upper()
  $$

In [0]:
%sql
Select *, databricks_cata.bronze.upper_func(brand) as brand_upper from products

In [0]:
# Saving Data Frame

df.write.format("delta")\
    .mode("overwrite")\
    .option("path","abfss://silver@databricksuk2025.dfs.core.windows.net/products")\
    .save()

In [0]:
# CREATING THE SILVER TABLES

%sql
Create table if not exists databricks_cata.silver.products_silver 
using DELTA
Location 'abfss://silver@databricksuk2025.dfs.core.windows.net/products'